In [9]:
import numpy as np
import matplotlib.pyplot as plt

kB = 8.617e-5        # Boltzmann constant in eV/K
T_REF = 873.15       # reference temperature = 600 °C, in kelvin

# ---------- The two "speeds" ----------
def reaction_rate(T_C, Ea, vR_ref, conc):
    """Growth rate (nm/min) if the SURFACE REACTION were the only limit.
    Arrhenius: rises exponentially with temperature."""
    T = T_C + 273.15
    return conc * vR_ref * np.exp(-Ea / kB * (1 / T - 1 / T_REF))

def transport_rate(T_C, vT_ref, pressure, flow, conc):
    """Growth rate (nm/min) if GAS DELIVERY were the only limit.
    Simplified textbook scalings:
      - gas diffusivity rises slowly with temperature (about T to the 1.75)
      - lower pressure means faster diffusion (1 / pressure)
      - faster flow thins the boundary layer (square root of flow)"""
    T = T_C + 273.15
    return conc * vT_ref * (T / T_REF) ** 1.75 * (1 / pressure) * np.sqrt(flow)

def growth_rate(T_C, Ea, vR_ref, vT_ref, pressure, flow, conc):
    """The two steps happen in series, like resistors, so the slower one dominates."""
    vR = reaction_rate(T_C, Ea, vR_ref, conc)
    vT = transport_rate(T_C, vT_ref, pressure, flow, conc)
    return 1 / (1 / vR + 1 / vT), vR, vT

# ---------- Everything that runs when you click the button ----------
def cvd_dashboard(T_op, Ea, vR_ref, vT_ref, pressure, flow, conc):
    T_grid = np.linspace(400, 1200, 400)                  # temperatures to plot (°C)
    v, vR, vT = growth_rate(T_grid, Ea, vR_ref, vT_ref, pressure, flow, conc)
    v_op, vR_op, vT_op = growth_rate(T_op, Ea, vR_ref, vT_ref, pressure, flow, conc)

    Da = vR_op / vT_op                                    # Damköhler number (k_s / h_G)
    share_reaction = (1 / vR_op) / (1 / vR_op + 1 / vT_op)  # how much of the "resistance" is the reaction

    if Da < 0.3:
        regime = "REACTION-LIMITED (the surface reaction is the bottleneck)"
    elif Da > 3:
        regime = "TRANSPORT-LIMITED (gas delivery is the bottleneck)"
    else:
        regime = "MIXED (both steps matter)"

    # crossover temperature, where the two speeds are equal (Da = 1)
    above = vR >= vT
    if above.all():
        cross_text, T_cross = "below 400 °C (transport-limited across the whole range)", None
    elif not above.any():
        cross_text, T_cross = "above 1200 °C (reaction-limited across the whole range)", None
    else:
        T_cross = T_grid[np.argmax(above)]
        cross_text = f"about {T_cross:.0f} °C"

    print(f"Operating point: {T_op:.0f} °C")
    print(f"Growth rate:                {v_op:.1f} nm/min")
    print(f"  if only reaction mattered: {vR_op:.1f} nm/min")
    print(f"  if only transport mattered: {vT_op:.1f} nm/min")
    print(f"Damköhler number (Da):      {Da:.2f}")
    print(f"Regime:                     {regime}")
    print(f"Share of the resistance:    reaction {100*share_reaction:.0f}%, transport {100*(1-share_reaction):.0f}%")
    print(f"Crossover temperature:      {cross_text}")

    # ---- what happens if you change one knob? ----
    def v_at(**changes):
        args = dict(T_C=T_op, Ea=Ea, vR_ref=vR_ref, vT_ref=vT_ref,
                    pressure=pressure, flow=flow, conc=conc)
        args.update(changes)
        return growth_rate(**args)[0]

    print("\nEffect on growth rate if you change ONE thing:")
    tests = [("Raise temperature by 50 °C", v_at(T_C=T_op + 50)),
             ("Double the gas flow", v_at(flow=flow * 2)),
             ("Halve the pressure", v_at(pressure=pressure / 2)),
             ("Double the precursor concentration", v_at(conc=conc * 2))]
    for label, new in tests:
        print(f"  {label:<36} {100 * (new / v_op - 1):+6.1f}%")

    # ---- plots ----
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))

    # left: the classic Arrhenius plot
    ax[0].semilogy(1000 / (T_grid + 273.15), vR, ":", label="Reaction only")
    ax[0].semilogy(1000 / (T_grid + 273.15), vT, ":", label="Transport only")
    ax[0].semilogy(1000 / (T_grid + 273.15), v, "k", linewidth=2, label="Combined (actual)")
    ax[0].plot(1000 / (T_op + 273.15), v_op, "ro", markersize=9, label="Your operating point")
    ax[0].set_xlabel("1000 / T  (1/K)   [hotter is to the left]")
    ax[0].set_ylabel("Growth rate (nm/min)")
    ax[0].set_title("Arrhenius plot: steep line, then a plateau")
    ax[0].legend()

    # right: the same thing against plain temperature
    ax[1].plot(T_grid, v, "k", linewidth=2, label="Combined (actual)")
    ax[1].plot(T_grid, vR, ":", label="Reaction only")
    ax[1].plot(T_grid, vT, ":", label="Transport only")
    ax[1].plot(T_op, v_op, "ro", markersize=9, label="Your operating point")
    if T_cross is not None:
        ax[1].axvline(T_cross, color="gray", linestyle="--", label="Crossover (Da = 1)")
    ax[1].set_ylim(0, 1.2 * vT.max())
    ax[1].set_xlabel("Temperature (°C)")
    ax[1].set_ylabel("Growth rate (nm/min)")
    ax[1].set_title("Growth rate vs temperature")
    ax[1].legend()

    plt.tight_layout()
    plt.show()

In [10]:
from ipywidgets import interact_manual, FloatSlider, FloatLogSlider

st = {"description_width": "190px"}
lay = {"width": "600px"}

interact_manual(
    cvd_dashboard,
    T_op=FloatSlider(value=650, min=400, max=1200, step=10,
                     description="Temperature (°C)", style=st, layout=lay),
    Ea=FloatSlider(value=1.6, min=0.5, max=3.0, step=0.1,
                   description="Activation energy (eV)", style=st, layout=lay),
    vR_ref=FloatLogSlider(value=5, base=10, min=-1, max=2, step=0.05,
                          description="Reaction speed at 600 °C", style=st, layout=lay),
    vT_ref=FloatLogSlider(value=200, base=10, min=1, max=3, step=0.05,
                          description="Gas delivery speed at 600 °C", style=st, layout=lay),
    pressure=FloatLogSlider(value=1, base=10, min=-1, max=1, step=0.05,
                            description="Pressure (relative)", style=st, layout=lay),
    flow=FloatLogSlider(value=1, base=10, min=-0.7, max=0.7, step=0.05,
                        description="Gas flow (relative)", style=st, layout=lay),
    conc=FloatLogSlider(value=1, base=10, min=-0.7, max=0.7, step=0.05,
                        description="Precursor concentration (relative)", style=st, layout=lay),
)

interactive(children=(FloatSlider(value=650.0, description='Temperature (°C)', layout=Layout(width='600px'), m…

<function __main__.cvd_dashboard(T_op, Ea, vR_ref, vT_ref, pressure, flow, conc)>

In [11]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- The model (Steps 1 to 4 from the report) ----------
def coverage(t, kp):
    """Step 1: fraction of surface sites occupied after a pulse of t seconds."""
    return 1 - np.exp(-kp * t)

def pulse_time_for(f, kp):
    """Step 2: pulse time needed to reach a target coverage f (between 0 and 1)."""
    return -np.log(1 - f) / kp

def process_numbers(f, kp, gpc_max, d_target, purge):
    """Steps 2 to 4 combined. Assumes both precursor pulses have the same length
    and that the second precursor saturates fully."""
    t_pulse = pulse_time_for(f, kp)                 # required pulse time (s)
    gpc = gpc_max * f                               # Step 3: growth per cycle (nm)
    n = int(np.ceil(d_target / gpc))                # Step 3: number of cycles (rounded up)
    t_cycle = 2 * t_pulse + 2 * purge               # Step 4: pulse A + purge + pulse B + purge
    t_total = n * t_cycle                           # Step 4: total process time (s)
    return dict(t_pulse=t_pulse, gpc=gpc, n=n, t_cycle=t_cycle, t_total=t_total,
                thickness=n * gpc, rate_nm_per_h=gpc / t_cycle * 3600)

# ---------- Everything that runs when you click the button ----------
def ald_dashboard(kp, f_percent, gpc_max, d_target, purge):
    f = f_percent / 100
    r = process_numbers(f, kp, gpc_max, d_target, purge)

    print(f"Target coverage:            {f_percent:.2f}%")
    print(f"Required pulse time:        {r['t_pulse']:.2f} s   (each precursor)")
    print(f"Growth per cycle:           {r['gpc']:.4f} nm")
    print(f"Cycles needed:              {r['n']}")
    print(f"Thickness actually reached: {r['thickness']:.2f} nm   (target {d_target} nm)")
    print(f"Time per cycle:             {r['t_cycle']:.2f} s   "
          f"(pulses {2*r['t_pulse']:.2f} s + purges {2*purge:.2f} s)")
    print(f"Total process time:         {r['t_total']/60:.1f} min")
    print(f"Average growth rate:        {r['rate_nm_per_h']:.0f} nm per hour")

    print("\nSame process at other target coverages:")
    print(f"  {'Coverage':>9} {'Pulse (s)':>10} {'Cycles':>7} {'Total (min)':>12}")
    for ff in (0.90, 0.99, 0.999):
        q = process_numbers(ff, kp, gpc_max, d_target, purge)
        print(f"  {100*ff:>8.1f}% {q['t_pulse']:>10.2f} {q['n']:>7d} {q['t_total']/60:>12.1f}")

    fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

    # Plot 1: saturation curve
    t = np.linspace(0, 10 / kp, 300)
    ax[0].plot(t, 100 * coverage(t, kp), "k", linewidth=2, label=f"kp = {kp:.2g} /s")
    ax[0].plot(t, 100 * coverage(t, 0.5 * kp), ":", label="half the fill rate")
    ax[0].plot(t, 100 * coverage(t, 2 * kp), ":", label="double the fill rate")
    ax[0].axhline(f_percent, color="gray", linestyle="--")
    ax[0].axvline(r["t_pulse"], color="gray", linestyle="--")
    ax[0].plot(r["t_pulse"], f_percent, "ro", markersize=9, label="Required pulse")
    ax[0].set_xlabel("Pulse time (s)")
    ax[0].set_ylabel("Surface coverage (%)")
    ax[0].set_title("Saturation curve")
    ax[0].legend()

    # Plot 2: thickness vs number of cycles
    cycles = np.arange(0, int(1.2 * r["n"]) + 1)
    ax[1].plot(cycles, cycles * r["gpc"], "k", linewidth=2)
    ax[1].axhline(d_target, color="gray", linestyle="--", label=f"Target {d_target} nm")
    ax[1].plot(r["n"], r["thickness"], "ro", markersize=9, label=f"{r['n']} cycles")
    ax[1].set_xlabel("Number of cycles")
    ax[1].set_ylabel("Film thickness (nm)")
    ax[1].set_title("Thickness is linear in cycles")
    ax[1].legend()

    # Plot 3: the tradeoff
    f_grid = np.linspace(0.5, 0.9999, 400)
    n_grid = np.ceil(d_target / (gpc_max * f_grid))
    t_grid = n_grid * (2 * pulse_time_for(f_grid, kp) + 2 * purge) / 60
    ax[2].plot(100 * f_grid, t_grid, "k", linewidth=2)
    ax[2].plot(f_percent, r["t_total"] / 60, "ro", markersize=9, label="Your target")
    ax[2].set_xlabel("Target coverage (%)")
    ax[2].set_ylabel("Total process time (min)")
    ax[2].set_title("Tradeoff: more saturation costs more time")
    ax[2].legend()

    plt.tight_layout()
    plt.show()

In [12]:
from ipywidgets import interact_manual, FloatSlider, FloatLogSlider

st = {"description_width": "230px"}
lay = {"width": "650px"}

interact_manual(
    ald_dashboard,
    kp=FloatLogSlider(value=3, base=10, min=-0.5, max=1.5, step=0.05,
                      description="Fill rate kp (1/s)", style=st, layout=lay),
    f_percent=FloatSlider(value=99, min=50, max=99.99, step=0.01,
                          description="Target coverage (%)", style=st, layout=lay),
    gpc_max=FloatSlider(value=0.1, min=0.05, max=0.3, step=0.005,
                        description="Max growth per cycle (nm)", style=st, layout=lay),
    d_target=FloatSlider(value=10, min=1, max=50, step=1,
                         description="Target thickness (nm)", style=st, layout=lay),
    purge=FloatSlider(value=1.5, min=0.2, max=10, step=0.1,
                      description="Purge time, each (s)", style=st, layout=lay),
)

interactive(children=(FloatLogSlider(value=3.0, description='Fill rate kp (1/s)', layout=Layout(width='650px')…

<function __main__.ald_dashboard(kp, f_percent, gpc_max, d_target, purge)>

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded

# ---------------------------------------------------------------
# THE MODEL IN ONE PARAGRAPH
# Gas enters at the trench opening (depth 0) and diffuses down to the bottom (depth 1).
# On the way it is used up by reaction on the side walls.
#   CVD: the walls keep reacting, so the gas profile is a fixed curve (closed-form cosh).
#   ALD: each wall site can only react once. As the upper walls fill up (coverage theta -> 1),
#        they stop using gas, so the gas reaches deeper. We step this forward in time.
#
# Two textbook kinetic-theory assumptions (NOT verified against a source, so check them):
#   diffusion in a narrow slit (Knudsen):   D   = (2/3) * w * v      (v = mean molecular speed)
#   wall reaction rate constant:            k_s = s * v / 4          (s = sticking probability)
# Putting them in the Thiele modulus  phi = L * sqrt(2 * k_s / (D * w))  gives
#   phi = AspectRatio * sqrt(0.75 * s)
# ---------------------------------------------------------------

def thiele(AR, s):
    """Thiele modulus: reaction speed compared with diffusion speed."""
    return AR * np.sqrt(0.75 * s)

def cvd_profile(xi, phi):
    """CVD: relative film growth along the trench (1 at the top). Closed-form result."""
    return np.cosh(phi * (1 - xi)) / np.cosh(phi)

def _bands(N):
    lower = np.ones(N - 1); lower[-1] = 2.0     # bottom boundary: no gas flows through the bottom
    upper = np.ones(N - 1); upper[0] = 0.0      # top boundary: gas concentration fixed at 1
    return lower, upper

def gas_profile(theta, phi, dxi, lower, upper):
    """Gas concentration down the trench for a given wall coverage theta (solved each time step).
    Balance of diffusion against consumption by the still-vacant wall sites (1 - theta)."""
    N = len(theta)
    main = -(2.0 + phi**2 * (1.0 - theta) * dxi**2)
    main[0] = 1.0
    ab = np.zeros((3, N))
    ab[0, 1:] = upper
    ab[1, :] = main
    ab[2, :-1] = lower
    rhs = np.zeros(N); rhs[0] = 1.0
    return solve_banded((1, 1), ab, rhs)

def simulate_ald(phi, kp, f, snap_times=(), N=None, max_steps=100000):
    """Step the ALD pulse forward in time until the BOTTOM of the trench reaches coverage f.
    phi = Thiele modulus, kp = flat-surface fill rate (1/s), f = target coverage (0 to 1)."""
    N = int(N or np.clip(8 * phi, 150, 800))
    dxi = 1.0 / (N - 1)
    lower, upper = _bands(N)
    theta = np.zeros(N)                       # wall coverage at each depth (starts empty)
    t = 0.0
    pending = sorted(snap_times)
    snaps = []
    hist_t, hist_b = [0.0], [0.0]
    for step in range(max_steps):
        c = gas_profile(theta, phi, dxi, lower, upper)
        rate = kp * c * (1.0 - theta)         # how fast each depth is filling right now
        dt = 0.05 / rate.max()                # small steps while things change quickly
        take_snap = False
        if pending and t + dt >= pending[0]:
            dt = pending[0] - t
            take_snap = True
        b_old, r_b = theta[-1], kp * c[-1]
        theta = 1.0 - (1.0 - theta) * np.exp(-kp * c * dt)     # exact filling over this step
        if theta[-1] >= f:                                     # bottom reached the target
            t_hit = t - np.log((1.0 - f) / (1.0 - b_old)) / r_b
            hist_t.append(t_hit); hist_b.append(f)
            return dict(converged=True, t_req=t_hit, xi=np.linspace(0, 1, N), theta=theta,
                        snaps=snaps, hist_t=hist_t, hist_b=hist_b, N=N)
        t += dt
        hist_t.append(t); hist_b.append(theta[-1])
        if take_snap:
            snaps.append((t, theta.copy())); pending.pop(0)
    return dict(converged=False, t_req=np.nan, xi=np.linspace(0, 1, N), theta=theta,
                snaps=snaps, hist_t=hist_t, hist_b=hist_b, N=N)

# ---------- Everything that runs when you click the button ----------
def trench_dashboard(AR, s, kappa1, f_percent):
    f = f_percent / 100
    phi = thiele(AR, s)
    kp = s * kappa1                           # flat-surface fill rate (same kp as Part B)
    t_flat = -np.log(1 - f) / kp              # Part B answer: pulse time on a flat surface
    cvd_sc = 1 / np.cosh(phi)                 # CVD step coverage

    print("Running the trench simulation...")
    res = simulate_ald(phi, kp, f)
    if not res["converged"]:
        print("The ALD simulation did not finish (too many steps). Try a lower aspect ratio or sticking probability.")
        return
    t_req = res["t_req"]
    bottom_at_flat = np.interp(t_flat, res["hist_t"], res["hist_b"])
    t_est = t_flat + 3 * AR**2 / (8 * kappa1)

    # check the numerical gas solver against the exact cosh formula (at t = 0)
    N = res["N"]; xi = res["xi"]; dxi = 1.0 / (N - 1)
    lo, up = _bands(N)
    c_num = gas_profile(np.zeros(N), phi, dxi, lo, up)
    c_exact = cvd_profile(xi, phi)

    print(f"Aspect ratio (depth ÷ width):             {AR:.0f}")
    print(f"Thiele modulus φ:                          {phi:.2f}")
    print(f"Flat-surface fill rate kp = s × κ1:        {kp:.2f} per second")
    print("\nCVD")
    print(f"  Step coverage (bottom ÷ top):            {100 * cvd_sc:.1f}%")
    print("\nALD")
    print(f"  Pulse time on a flat surface (Part B):   {t_flat:.2f} s")
    print(f"  Pulse time to coat the trench bottom:    {t_req:.2f} s   ({t_req / t_flat:.1f}× the flat-surface time)")
    print(f"  Bottom coverage if you used the flat-surface pulse: {100 * bottom_at_flat:.1f}%")
    print(f"  Rough estimate (flat time + 3·AR²/(8·κ1)): {t_est:.2f} s")
    print(f"\nCheck: numerical gas profile vs exact formula at t = 0, max difference {np.max(np.abs(c_num - c_exact)):.4f}")

    # snapshots of the ALD coverage profile at earlier times
    res2 = simulate_ald(phi, kp, f, snap_times=[0.1 * t_req, 0.3 * t_req, 0.6 * t_req], N=N)

    fig, ax = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: how the film is distributed down the trench
    for (ts, th) in res2["snaps"]:
        ax[0].plot(xi, th, ":", label=f"ALD, t = {ts:.2f} s")
    ax[0].plot(xi, res2["theta"], "k", linewidth=2, label=f"ALD, t = {t_req:.2f} s (required)")
    ax[0].plot(xi, c_exact, "r--", linewidth=2, label=f"CVD (bottom ÷ top = {100 * cvd_sc:.0f}%)")
    ax[0].set_xlabel("Depth into trench (0 = opening, 1 = bottom)")
    ax[0].set_ylabel("Relative film thickness (ALD: coverage θ)")
    ax[0].set_title("Film thickness down the trench")
    ax[0].legend()

    # Plot 2: CVD step coverage vs aspect ratio
    AR_line = np.logspace(0, 2, 200)
    for sj in (s / 10, s, min(1.0, s * 10)):
        ax[1].semilogx(AR_line, 1 / np.cosh(thiele(AR_line, sj)), label=f"CVD, s = {sj:.2g}")
    ax[1].axhline(1.0, color="gray", linestyle="--", label="ALD (pulse long enough)")
    ax[1].plot(AR, cvd_sc, "ro", markersize=9, label="Your setting (CVD)")
    ax[1].set_ylim(0, 1.05)
    ax[1].set_xlabel("Aspect ratio (depth ÷ width)")
    ax[1].set_ylabel("Step coverage (bottom ÷ top)")
    ax[1].set_title("CVD coverage falls as trenches get deeper")
    ax[1].legend()

    # Plot 3: ALD pulse time vs aspect ratio (this sweep can take a while)
    print("Running the aspect-ratio sweep for the right-hand plot...")
    AR_grid = np.union1d(np.unique(np.round(np.logspace(0, 2, 11)).astype(int)), [int(round(AR))])
    t_list = []
    for a in AR_grid:
        pj = thiele(a, s)
        r = simulate_ald(pj, kp, f, N=int(np.clip(6 * pj, 100, 400)), max_steps=60000)
        t_list.append(r["t_req"])
    ax[2].loglog(AR_grid, t_list, "ko-", label="Simulation")
    ax[2].axhline(t_flat, color="gray", linestyle=":", label="Flat-surface time (Part B)")
    ax[2].loglog(AR_line, t_flat + 3 * AR_line**2 / (8 * kappa1), "r--", label="Rough estimate: flat time + 3·AR²/(8·κ1)")
    ax[2].plot(AR, t_req, "ro", markersize=9, label="Your setting")
    ax[2].set_xlabel("Aspect ratio (depth ÷ width)")
    ax[2].set_ylabel("Pulse time to coat the bottom (s)")
    ax[2].set_title("ALD pulse time grows with aspect ratio")
    ax[2].legend()

    plt.tight_layout()
    plt.show()


In [14]:
from ipywidgets import interact_manual, FloatSlider, FloatLogSlider

st = {"description_width": "280px"}
lay = {"width": "720px"}

interact_manual(
    trench_dashboard,
    AR=FloatSlider(value=20, min=1, max=100, step=1,
                   description="Aspect ratio (depth ÷ width)", style=st, layout=lay),
    s=FloatLogSlider(value=0.01, base=10, min=-4, max=-0.5, step=0.05,
                     description="Sticking probability s", style=st, layout=lay),
    kappa1=FloatLogSlider(value=300, base=10, min=1, max=4, step=0.05,
                          description="Impingement rate per site κ1 (1/s)", style=st, layout=lay),
    f_percent=FloatSlider(value=99, min=50, max=99.99, step=0.01,
                          description="Target coverage at the bottom (%)", style=st, layout=lay),
)

interactive(children=(FloatSlider(value=20.0, description='Aspect ratio (depth ÷ width)', layout=Layout(width=…

<function __main__.trench_dashboard(AR, s, kappa1, f_percent)>